# 세그먼트(2축·7축) + 필터·랭킹 비교 — 입력값은 맨 아래에서

`customers_mvno.csv` + `통신요금제_통합데이터_최종.csv`로 돌아가는 버전.

같은 입력 하나를 넣으면 네 가지를 순서대로 비교해서 보여준다.

1. 세그먼트(필수 2축)가 많이 쓰는 요금제
2. 필수 2축(예산·데이터) 조건으로 직접 필터링한 요금제 (요금 낮은 순)
3. 세그먼트(전체 7축)가 많이 쓰는 요금제
4. 전체 조건(예산·데이터·통화·문자·OTT)으로 직접 필터링한 요금제 (요금 낮은 순)

### 참고
- 데이터량은 `band_for()`가 없어서 구간(band) 대신 GB 숫자를 그대로 축으로 씀
- 요금제명은 `plan_id` 기준으로 카탈로그와 조인해서 보여줌

### 쓰는 법
위에서부터 순서대로 한 번씩 실행한 다음, **맨 아래 `MY_INPUT` 셀만** 계속 바꿔가며
재실행하면 됨. 그 위 셀들(모델 학습 포함)은 한 번만 실행하면 다시 안 돌려도 됨.

In [41]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

SEED = 20260812
K = 6                          # 원본 노트북에서 실루엣 기준으로 고른 값
TOP_N = 5
UNLIMITED_SENTINEL = 9999.0    # 통화/문자 무제한을 숫자로 표현할 때 쓰는 값
DATA_UNLIMITED_GB = 300.0      # 데이터 무제한을 GB 숫자로 표현할 때 쓰는 값 (임의 큰 값)

CUSTOMERS_PATH = "data/synthetic/customers_mvno.csv"
PLANS_PATH = "data/final/통신요금제_통합데이터_최종.csv"

REQUIRED_AXES = ["예산", "데이터GB"]
ALL_AXES = ["예산", "데이터GB", "통화분수", "문자건수", "나이", "OTT희망", "OTT필수"]

## 1. 요금제 카탈로그 로드

MVNO 요금제만 남기고, `plan_id`를 문자열로 통일해서 나중에 `source_plan_id`와 조인할 수 있게 함.

In [42]:
plans = pd.read_csv(PLANS_PATH, encoding="utf-8-sig")
plans = plans[plans["carrier_type"].eq("MVNO")].copy()
plans["plan_id"] = plans["plan_id"].astype(str)
plans = plans.set_index("plan_id")

# 결과 표에 보여줄 열만 추림
PLAN_DISPLAY_COLS = ["mvno_brand", "plan_name", "data_gb", "data_unlimited",
                      "voice_unlimited", "voice_minutes", "monthly_fee", "discounted_fee"]

print(f"MVNO 요금제 {len(plans):,}개 로드")
plans[PLAN_DISPLAY_COLS].head(3)

MVNO 요금제 2,278개 로드


,mvno_brand,plan_name,data_gb,data_unlimited,voice_unlimited,voice_minutes,monthly_fee,discounted_fee
plan_id,,,,,,,,
36334,LG U+,너겟49,120.0,False,True,NaN,7000,7000
26257,U+유모바일,[평생할인] 7GB+/통화기본,7.0,False,True,NaN,15900,0
36971,KT엠모바일,5G 모두다 맘껏 안심 20GB+,20.0,False,True,NaN,19900,0


## 2. 축 만드는 함수

고객(또는 사용자 입력 1건)을 KMeans가 쓸 숫자 축으로 바꾼다.

In [43]:
def make_axes(df: pd.DataFrame) -> pd.DataFrame:
    """고객(또는 입력 1건)을 KMeans가 쓸 숫자 축으로 변환."""
    data_gb = np.where(
        df["data_unlimited_need"].astype(bool),
        DATA_UNLIMITED_GB,
        df["data_gb_month"].fillna(0.0),
    )
    return pd.DataFrame({
        "예산": df["budget_krw"].to_numpy(dtype=float),
        "데이터GB": data_gb,
        "통화분수": df["voice_minutes_need"].fillna(UNLIMITED_SENTINEL).to_numpy(dtype=float),
        "문자건수": df["sms_count_need"].fillna(UNLIMITED_SENTINEL).to_numpy(dtype=float),
        "나이": df["age"].to_numpy(dtype=float),
        "OTT희망": df["ott_want"].notna().astype(int).to_numpy(),
        "OTT필수": df["ott_required"].astype(int).to_numpy(),
    }, index=df.index)

## 3. 세그먼트 모델

`fit`: 고객 전체를 K개 덩어리로 나누고, 덩어리마다 실제로 그 사람들이 쓰는 요금제
(`source_plan_id`) 상위 5개를 기억한다.

`explain`: 이제 `plan_id`를 카탈로그(`plans`)에 조인해서 **요금제명·통신사·요금·데이터량**까지 같이 보여준다.

In [44]:
class SegmentModel:
    """KMeans 군집 -> 그 세그먼트 사람들이 실제로 쓰는 요금제(source_plan_id) 상위 TOP_N."""

    def __init__(self, axes: list, name: str):
        self.axes = axes
        self.name = name

    def fit(self, df: pd.DataFrame) -> "SegmentModel":
        x = make_axes(df)[self.axes]
        self.pipe = make_pipeline(StandardScaler(), KMeans(K, random_state=SEED, n_init=10))
        labels = self.pipe.fit_predict(x)
        self.labels_ = labels
        self.plans_by_segment = {
            seg: df.loc[labels == seg, "source_plan_id"].value_counts().head(TOP_N)
            for seg in range(K)
        }
        self.profile_ = df.assign(세그먼트=labels).groupby("세그먼트").agg(
            인원=("customer_id", "size"),
            평균예산=("budget_krw", "mean"),
            평균GB=("data_gb_month", "mean"),
            평균나이=("age", "mean"),
            통화무제한비율=("voice_unlimited_need", "mean"),
        ).round(1)
        return self

    def segment_of(self, one_row_df: pd.DataFrame) -> int:
        x = make_axes(one_row_df)[self.axes]
        return int(self.pipe.predict(x)[0])

    def explain(self, seg: int):
        print(f"[{self.name}] -> 세그먼트 {seg}")
        display(self.profile_.loc[[seg]])

        top = self.plans_by_segment[seg]
        n = self.profile_.loc[seg, "인원"]

        rows = []
        for plan_id, cnt in top.items():
            info = plans.loc[plan_id, PLAN_DISPLAY_COLS] if plan_id in plans.index else None
            row = {"plan_id": plan_id, "인원": cnt, "세그먼트 내 비율": f"{cnt / n * 100:.1f}%"}
            if info is not None:
                row.update({
                    "통신사(알뜰폰)": info["mvno_brand"],
                    "요금제명": info["plan_name"],
                    "데이터": "무제한" if info["data_unlimited"] else f"{info['data_gb']}GB",
                    "통화": "무제한" if info["voice_unlimited"] else info["voice_minutes"],
                    "월요금(할인후)": info["discounted_fee"],
                })
            else:
                row["요금제명"] = "(카탈로그에 없음)"
            rows.append(row)

        result = pd.DataFrame(rows).set_index("plan_id")
        print(f"이 세그먼트가 실제로 많이 쓰는 요금제 (상위 {TOP_N}):")
        display(result)
        return result

## 4. 데이터 로드 & 모델 학습

한 번만 실행하면 됨 (아래부터는 입력값만 바꿔가며 재실행).

In [45]:
customers = pd.read_csv(CUSTOMERS_PATH, encoding="utf-8-sig")
customers["source_plan_id"] = customers["source_plan_id"].astype(str)
print(f"합성 고객 {len(customers):,}명 로드")

seg_req = SegmentModel(REQUIRED_AXES, "세그먼트(필수 2축: 예산·데이터)").fit(customers)
seg_all = SegmentModel(ALL_AXES, "세그먼트(전체 7축)").fit(customers)
print("모델 학습 완료")

합성 고객 40,000명 로드
모델 학습 완료


## 4. 세그먼트 전체 구성 확인

K=6개 세그먼트 전부가 각각 어떤 사람들이고 어떤 요금제를 쓰는지 본다.

In [46]:
def show_all_segments(model: SegmentModel):
    print(f"===== {model.name} =====")
    display(model.profile_.sort_index())

    for seg in sorted(model.plans_by_segment):
        n = model.profile_.loc[seg, "인원"]
        top = model.plans_by_segment[seg]
        print(f"\n[세그먼트 {seg}] 인원 {n:,}명 — 자주 쓰는 요금제 Top{TOP_N}")

        rows = []
        for plan_id, cnt in top.items():
            info = plans.loc[plan_id, PLAN_DISPLAY_COLS] if plan_id in plans.index else None
            row = {"plan_id": plan_id, "인원": cnt, "세그먼트 내 비율": f"{cnt / n * 100:.1f}%"}
            if info is not None:
                row.update({
                    "통신사": info["mvno_brand"],
                    "요금제명": info["plan_name"],
                    "데이터": "무제한" if info["data_unlimited"] else f"{info['data_gb']}GB",
                    "월요금": info["discounted_fee"],
                })
            rows.append(row)
        display(pd.DataFrame(rows).set_index("plan_id"))


show_all_segments(seg_req)   # 2축
show_all_segments(seg_all)   # 7축

===== 세그먼트(필수 2축: 예산·데이터) =====


,인원,평균예산,평균GB,평균나이,통화무제한비율
세그먼트,,,,,
0,10844,9287.0,10.9,46.7,0.8
1,9537,12774.3,92.4,46.1,1.0
2,1085,40930.7,99.6,46.5,1.0
3,14286,1028.4,8.9,50.8,0.4
4,1814,24935.2,104.8,46.2,1.0
5,2434,24379.0,11.4,46.0,0.8



[세그먼트 0] 인원 10,844명 — 자주 쓰는 요금제 Top5


,인원,세그먼트 내 비율,통신사,요금제명,데이터,월요금
plan_id,,,,,,
28127,1353,12.5%,마블링,[모요only] 마블링 프라임 11GB+매일2GB+3Mbps,11.0GB,6700
33042,807,7.4%,이지모바일,EG 무제한 7GB+1M(밀리의 서재),7.0GB,5900
23975,758,7.0%,인스모바일,인스 유심 스트롱 11GB+,11.0GB,8750
34377,636,5.9%,찬스모바일,[모요only] 알찬 음성기본 11GB+일 2GB+3Mbps,11.0GB,7800
34917,579,5.3%,더원모바일,ONE 5G 30GB 기본,30.0GB,15400



[세그먼트 1] 인원 9,537명 — 자주 쓰는 요금제 Top5


...

,인원,세그먼트 내 비율,통신사,요금제명,데이터,월요금
plan_id,,,,,,
28127,654,9.7%,마블링,[모요only] 마블링 프라임 11GB+매일2GB+3Mbps,11.0GB,6700
30830,555,8.2%,마블링,[모요only] 마블링 프라임 7GB+1Mbps,7.0GB,3700
23450,469,7.0%,KT스카이라이프,통화 충분 15GB,15.0GB,0
26257,403,6.0%,U+유모바일,[평생할인] 7GB+/통화기본,7.0GB,0
23975,370,5.5%,인스모바일,인스 유심 스트롱 11GB+,11.0GB,8750



[세그먼트 4] 인원 10,220명 — 자주 쓰는 요금제 Top5


,인원,세그먼트 내 비율,통신사,요금제명,데이터,월요금
plan_id,,,,,,
28102,1215,11.9%,핀다이렉트,[모요only] [K] 핀다이렉트Z Max (100GB) _PIN,100.0GB,15800
26613,1124,11.0%,쉐이크모바일,[모요only] [모요핫딜] 쉐이크 LTE 100GB+Npay 매월 5천원...,100.0GB,11700
37103,770,7.5%,핀다이렉트,[모요only] [K] 핀다이렉트Z 100GB+(네이버페이) _모요핫딜...,100.0GB,11700
30954,750,7.3%,핀다이렉트,[모요only] [K] 핀다이렉트Z 100GB+(네이버페이) _PIN...,100.0GB,11700
35834,414,4.1%,핀다이렉트,[모요only] [K] 핀다이렉트Z Max (100GB)(12) _PIN,100.0GB,24000



[세그먼트 5] 인원 3,205명 — 자주 쓰는 요금제 Top5


,인원,세그먼트 내 비율,통신사,요금제명,데이터,월요금
plan_id,,,,,,
33042,411,12.8%,이지모바일,EG 무제한 7GB+1M(밀리의 서재),7.0GB,5900
36163,336,10.5%,쉐이크모바일,[모요only] 쉐이크only LTE 100GB+밀리의서재,100.0GB,11900
31929,253,7.9%,이지모바일,EG 무제한 100GB+5M(밀리의 서재),100.0GB,14900
37221,236,7.4%,에이모바일,A 스페셜 100GB+,100.0GB,11000
37435,193,6.0%,에이모바일,A 스페셜 11GB+,11.0GB,6700


## 5. 필터 + 랭킹 함수

세그먼트를 거치지 않고, 조건으로 직접 걸러서 요금 낮은 순으로 정렬하는 방식.

In [47]:
def filter_plans_required(catalog: pd.DataFrame, q: dict) -> pd.DataFrame:
    """필수 2축(예산·데이터)만으로 필터링, 요금 낮은 순 정렬."""
    df = catalog.copy()
    df = df[df["discounted_fee"] <= q["budget_krw"]]
    if q.get("data_unlimited_need"):
        df = df[df["data_unlimited"]]
    else:
        need_gb = q.get("data_gb_month", 0.0)
        df = df[df["data_unlimited"] | (df["data_gb"] >= need_gb)]
    return df.sort_values("discounted_fee", ascending=True)


def filter_plans_all(catalog: pd.DataFrame, q: dict) -> pd.DataFrame:
    """전체 조건(예산·데이터·통화·문자·OTT)으로 필터링, 요금 낮은 순 정렬."""
    df = filter_plans_required(catalog, q)  # 2축 필터를 먼저 걸고 그 위에 추가 조건
    if q.get("voice_unlimited_need"):
        df = df[df["voice_unlimited"]]
    elif q.get("voice_minutes_need") is not None:
        df = df[df["voice_unlimited"] | (df["voice_minutes"] >= q["voice_minutes_need"])]
    if q.get("sms_unlimited_need"):
        df = df[df["sms_unlimited"]]
    elif q.get("sms_count_need") is not None:
        df = df[df["sms_unlimited"] | (df["sms_count"] >= q["sms_count_need"])]
    if q.get("ott_required") and q.get("ott_want"):
        df = df[df["ott_options"].fillna("").str.contains(q["ott_want"], regex=False)]
    return df.sort_values("discounted_fee", ascending=True)

## 6. 입력 -> 네 가지 결과 비교 함수

In [48]:
def query_to_row(q: dict) -> pd.DataFrame:
    """사용자 입력 dict -> make_axes가 먹는 1행짜리 DataFrame."""
    return pd.DataFrame([{
        "budget_krw": q["budget_krw"],
        "data_gb_month": q.get("data_gb_month", 0.0),
        "data_unlimited_need": q.get("data_unlimited_need", False),
        "voice_minutes_need": q.get("voice_minutes_need", np.nan),
        "sms_count_need": q.get("sms_count_need", np.nan),
        "age": q["age"],
        "ott_want": q.get("ott_want"),
        "ott_required": q.get("ott_required", False),
    }])

In [49]:
def compare_all(seg_req, seg_all, q: dict):
    """[2축 세그먼트] -> [2축 필터+랭킹] -> [7축 세그먼트] -> [전체 필터+랭킹] 순으로 출력."""

    print("=" * 60)
    print("입력값")
    print("=" * 60)
    for k, v in q.items():
        print(f"  {k}: {v}")
    row = query_to_row(q)

    print("\n" + "=" * 60)
    print("[1] 세그먼트 - 필수 2축 (많이 쓰는 요금제)")
    print("=" * 60)
    seg_req.explain(seg_req.segment_of(row))

    print("=" * 60)
    print("[2] 필터 + 랭킹 - 필수 2축 조건만 (요금 낮은 순)")
    print("=" * 60)
    req_matched = filter_plans_required(plans, q)
    print(f"조건에 맞는 요금제 {len(req_matched):,}개\n")
    display(req_matched[PLAN_DISPLAY_COLS].head(10))

    print("\n" + "=" * 60)
    print("[3] 세그먼트 - 전체 7축 (많이 쓰는 요금제)")
    print("=" * 60)
    seg_all.explain(seg_all.segment_of(row))

    print("=" * 60)
    print("[4] 필터 + 랭킹 - 전체 조건 (요금 낮은 순)")
    print("=" * 60)
    all_matched = filter_plans_all(plans, q)
    print(f"조건에 맞는 요금제 {len(all_matched):,}개\n")
    display(all_matched[PLAN_DISPLAY_COLS].head(10))

    return req_matched, all_matched

## 7. 입력값 넣고 결과 보기

**여기 아래 셀만 값 바꿔가며 재실행(Shift+Enter)하면 됨.** 위 셀들은 다시 안 돌려도 됨.

- `data_unlimited_need=True`면 `data_gb_month` 값은 무시됨
- 통화/문자 무제한이면 해당 `_need` 값을 `None`으로, `_unlimited_need=True`로
- `ott_want`는 희망하는 OTT/부가서비스 이름 문자열 (없으면 `None`)

In [50]:
MY_INPUT = {
    "budget_krw": 15000,
    "data_gb_month": 30.0,
    "data_unlimited_need": False,
    "voice_minutes_need": 100.0,   # 무제한이면 None
    "voice_unlimited_need": False,
    "sms_count_need": 100.0,       # 무제한이면 None
    "sms_unlimited_need": False,
    "age": 31,
    "ott_want": None,              # 예: "밀리의서재" / 없으면 None
    "ott_required": False,
}

req_matched, all_matched = compare_all(seg_req, seg_all, MY_INPUT)

입력값
  budget_krw: 15000
  data_gb_month: 30.0
  data_unlimited_need: False
  voice_minutes_need: 100.0
  voice_unlimited_need: False
  sms_count_need: 100.0
  sms_unlimited_need: False
  age: 31
  ott_want: None
  ott_required: False

[1] 세그먼트 - 필수 2축 (많이 쓰는 요금제)
[세그먼트(필수 2축: 예산·데이터)] -> 세그먼트 0


,인원,평균예산,평균GB,평균나이,통화무제한비율
세그먼트,,,,,
0,10844,9287.0,10.9,46.7,0.8


이 세그먼트가 실제로 많이 쓰는 요금제 (상위 5):


,인원,세그먼트 내 비율,통신사(알뜰폰),요금제명,데이터,통화,월요금(할인후)
plan_id,,,,,,,
28127,1353,12.5%,마블링,[모요only] 마블링 프라임 11GB+매일2GB+3Mbps,11.0GB,무제한,6700
33042,807,7.4%,이지모바일,EG 무제한 7GB+1M(밀리의 서재),7.0GB,무제한,5900
23975,758,7.0%,인스모바일,인스 유심 스트롱 11GB+,11.0GB,무제한,8750
34377,636,5.9%,찬스모바일,[모요only] 알찬 음성기본 11GB+일 2GB+3Mbps,11.0GB,무제한,7800
34917,579,5.3%,더원모바일,ONE 5G 30GB 기본,30.0GB,무제한,15400


[2] 필터 + 랭킹 - 필수 2축 조건만 (요금 낮은 순)
조건에 맞는 요금제 65개



...

,인원,평균예산,평균GB,평균나이,통화무제한비율
세그먼트,,,,,
1,11775,3508.1,10.2,49.7,0.0


이 세그먼트가 실제로 많이 쓰는 요금제 (상위 5):


,인원,세그먼트 내 비율,통신사(알뜰폰),요금제명,데이터,통화,월요금(할인후)
plan_id,,,,,,,
23771,830,7.0%,티플러스,티플 가성비(300분/6GB)_평생요금,6.0GB,300.0,1900
23770,826,7.0%,티플러스,티플 가성비(300분/6GB)_평생요금,6.0GB,300.0,1900
26798,520,4.4%,티플러스,가성비플러스(15GB/350분),15.0GB,350.0,10
24201,520,4.4%,고고모바일,[모요only] 5G 200분/15GB,15.0GB,200.0,10
26836,490,4.2%,티플러스,가성비플러스(15GB/350분),15.0GB,350.0,10


[4] 필터 + 랭킹 - 전체 조건 (요금 낮은 순)
조건에 맞는 요금제 59개



,mvno_brand,plan_name,data_gb,data_unlimited,voice_unlimited,voice_minutes,monthly_fee,discounted_fee
plan_id,,,,,,,,
34304,티플러스,가성비플러스(30GB/350분),30.0,False,False,350.0,39600,4900
33129,티플러스,5G 가성비플러스(30GB/350분),30.0,False,False,350.0,39600,4900
37403,핀다이렉트,[모요only] [K] 핀다이렉트Z 100GB+(네이버페이) _PAY...,100.0,False,True,NaN,36900,6900
36334,LG U+,너겟49,120.0,False,True,NaN,7000,7000
37436,에이모바일,A 5G 스페셜 슈퍼,95.0,False,True,NaN,48420,7900
37442,핀다이렉트,[모요only] [S] 핀다이렉트Z Max(네이버페이) _PAY,100.0,False,True,NaN,42900,8100
37437,에이모바일,A 5G 스페셜 에센셜,125.0,False,True,NaN,50200,8400
36985,아이즈모바일,[L]5G 아이즈95GB+3Mbps,95.0,False,True,NaN,53900,8500
36341,티플러스,5G 가성비플러스(30GB/350분)_12개월할인,30.0,False,False,350.0,39600,8900


In [52]:
import numpy as np, pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler


pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)
pd.set_option("display.unicode.east_asian_width", True)   # 한글 정렬 필수

df = pd.read_csv(PLANS_PATH, encoding="utf-8").reset_index(drop=True)
df["final_price"] = np.where(df["discounted_fee"] > 0, df["discounted_fee"], df["monthly_fee"])

UNLIM_D, UNLIM_V, UNLIM_S = 300.0, 3000.0, 3000.0
df["f_data"]  = np.where(df["data_unlimited"],  UNLIM_D, df["data_gb"].fillna(0))
df["f_voice"] = np.where(df["voice_unlimited"], UNLIM_V, df["voice_minutes"].fillna(0))
df["f_sms"]   = np.where(df["sms_unlimited"],   UNLIM_S, df["sms_count"].fillna(0))

user = dict(budget_krw=15000, data_gb_month=30.0,
            voice_minutes_need=100.0, sms_count_need=100.0)

FEATS = ["final_price", "f_data", "f_voice", "f_sms"]
uraw = np.array([[user["budget_krw"], user["data_gb_month"],
                  user["voice_minutes_need"], user["sms_count_need"]]], float)

sc = MinMaxScaler().fit(np.vstack([df[FEATS].values, uraw]))
P, u = sc.transform(df[FEATS].values), sc.transform(uraw)

df["cos_sim"] = cosine_similarity(u, P)[0]
df["euclid"]  = np.linalg.norm(P - u, axis=1)


# ------------------------------------------------------------------
def _fmt(v, unlim_at, unit):
    if v >= unlim_at:
        return "무제한"
    return f"{v:,.0f}{unit}" if float(v).is_integer() else f"{v:,.1f}{unit}"


def _verdict(r, user):
    f = []
    if r["final_price"] > user["budget_krw"]:     f.append("예산초과")
    if r["f_data"]  < user["data_gb_month"]:      f.append("데이터부족")
    if r["f_voice"] < user["voice_minutes_need"]: f.append("통화부족")
    if r["f_sms"]   < user["sms_count_need"]:     f.append("문자부족")
    return ", ".join(f) if f else "OK"


def make_table(sub, score_col, user) -> pd.DataFrame:
    out = pd.DataFrame({
        "통신사": sub["host_mno"].values,
        "브랜드": sub["mvno_brand"].fillna("-").values,
        "요금제": sub["plan_name"].str.slice(0, 30).values,
        "월요금": [f"{v:,}" for v in sub["final_price"]],
        "데이터": [_fmt(v, UNLIM_D, "GB") for v in sub["f_data"]],
        "통화":   [_fmt(v, UNLIM_V, "분") for v in sub["f_voice"]],
        "문자":   [_fmt(v, UNLIM_S, "건") for v in sub["f_sms"]],
        "점수":   sub[score_col].round(4).values,
        "판정":   [_verdict(r, user) for _, r in sub.iterrows()],
    })
    out.index = pd.RangeIndex(1, len(out) + 1, name="순위")
    return out


def style_table(tbl, caption):
    """주피터용: 판정 색상 + 점수 그라데이션"""
    return (tbl.style
            .set_caption(caption)
            .map(lambda v: "color:#1a7f37;font-weight:600" if v == "OK"
                 else "color:#c0392b", subset=["판정"])
            .background_gradient(subset=["점수"], cmap="Blues"))


def summary_row(sub, user, label):
    v = [_verdict(r, user) for _, r in sub.iterrows()]
    return {"방식": label,
            "충족": f"{sum(x == 'OK' for x in v)}/{len(v)}",
            "데이터부족": sum("데이터부족" in x for x in v),
            "예산초과": sum("예산초과" in x for x in v),
            "평균월요금": f"{sub['final_price'].mean():,.0f}",
            "점수범위": f"{sub.iloc[0]['_s']:.4f} ~ {sub.iloc[-1]['_s']:.4f}"}


# ------------------------------------------------------------------
TOP_N = 10
cos_top = df.nlargest(TOP_N, "cos_sim").assign(_s=lambda d: d["cos_sim"])
euc_top = df.nsmallest(TOP_N, "euclid").assign(_s=lambda d: d["euclid"])

tbl_cos = make_table(cos_top, "cos_sim", user)
tbl_euc = make_table(euc_top, "euclid", user)
cmp = pd.DataFrame([summary_row(cos_top, user, "cosine"),
                    summary_row(euc_top, user, "euclidean")]).set_index("방식")

print(f"[요구조건] 예산 {user['budget_krw']:,}원 · 데이터 {user['data_gb_month']:.0f}GB · "
      f"통화 {user['voice_minutes_need']:.0f}분 · 문자 {user['sms_count_need']:.0f}건")
print(f"[대상] 전체 요금제 {len(df):,}건\n")
print("=== 코사인 유사도 Top-10 ===");  print(tbl_cos, "\n")
print("=== 유클리드 거리 Top-10 ===");  print(tbl_euc, "\n")


[요구조건] 예산 15,000원 · 데이터 30GB · 통화 100분 · 문자 100건
[대상] 전체 요금제 2,826건

=== 코사인 유사도 Top-10 ===
     통신사        브랜드                                 요금제  월요금 데이터   통화   문자    점수                  판정
순위                                                                                                                       
1        KT    핀다이렉트  [모요only] [K] 핀다이렉트 Speed 30GB/  14,000   30GB  200분  100건  0.9978                    OK
2       SKT        프리티                  (5G)더든든한200분30GB  17,600   30GB  200분  100건  0.9965              예산초과
3        KT    KT엠모바일                 데이터 충분 25GB/100분  15,400   25GB  100분  100건  0.9960  예산초과, 데이터부족
4       SKT        프리티                  (5G)더든든한200분25GB  15,400   25GB  200분  100건  0.9944  예산초과, 데이터부족
5       SKT        프리티                  (5G)하나은행 200분30G  12,100   30GB  200분  100건  0.9923                    OK
6        KT    KT엠모바일                 데이터 충분 20GB/100분  13,400   20GB  100분  100건  0.9919            데이터부족
7      LGU+  아이즈모바일                 